In [6]:
## IMPORTS AND SETUP

# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)
import logging
import warnings
import tensorflow as tf
from dotenv import load_dotenv
from Leyanda_Project.models.callbacks import ConfusionMatrixCallback, create_callbacks
from Leyanda_Project.models.cnn_classifier import create_model, train_model, add_class_weights
from Leyanda_Project.preprocessing.binary_converter import convert_to_binary_dataset_structure
from Leyanda_Project.preprocessing.data_format import data_formats_fixes
from Leyanda_Project.preprocessing.data_loader import dataset_assembly, dataset_split
from Leyanda_Project.utils.naming import generate_model_name, get_model_path
from Leyanda_Project.utils.visualization import visualize_class_samples, visualize_class_distribution

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0=default, 1=info, 2=warning, 3=error
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Force to use only GPU 0
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Prevent TF from grabbing all GPU memory
warnings.filterwarnings('ignore', category=Warning) # Python warnings
logging.root.removeHandler(logging.root.handlers) # Disable ABSL logging
logging.getLogger('absl').propagate = False
logging.getLogger('tensorflow').propagate = False
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
load_dotenv("/tf/projet/.env")
WANDB_API_KEY = os.getenv("API_KEY")
wandb_entity = "tom-antoine-cesi"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
GPU is available: 1 device(s) detected


In [7]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
model_arch = "CNN"

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset" # Path to the raw data folder
data_format_fix = False # Set to True to fix data formats
data_visualization = True # Set to True to visualize data
excluded_data_folders = ["Dataset Livrable 2"] # Folders to exclude from the dataset
batch_size = 64 # Batch size for dataset loading
img_height = 180 # Image height for dataset loading
img_width = 180 # Image width for dataset loading
image_size = f"{img_height}x{img_width}"

# Dataset split parameters
train_split = 0.8 # Proportion of the dataset to use for training
val_split = 0.1 # Proportion of the dataset to use for validation
test_split = 0.1 # Proportion of the dataset to use for testing

# Model configuration parameters
learning_rate = 0.001
epochs = 10
transfer_learning = True # Set to True to create a VGG16 model with custom top, False to create a custom model
train_transfer_model = True # Set to True to train the VGG-16 model
class_weight = True # Set to True to use class weights for imbalanced datasets
target_binary_class_name = "Photo" # Name of the target binary class for binary classification ("Photo", "Painting"...), None for multi-class
save_path = "/tf/projet/Livrable 1/models" # Path to the models folder
binary_dataset_folder_output = f"/tf/projet/Dataset_binary_{target_binary_class_name}" # Path to the binary dataset folder

In [8]:
model_name = generate_model_name(
    project_name=project_name,
    model_arch=model_arch,
    target_class=target_binary_class_name if target_binary_class_name else None,
    transfer_learning=transfer_learning,
    epoch=epochs,
    batch_size=batch_size,
    class_weight=class_weight,
    train_transfer_model=train_transfer_model
)


--Generating model name--
Generated model name: Leyanda_CNN_bin_Photo_transfer_e10_b64_class_weight_fine_tuning
